In [2]:
import tensorflow as tf
import time

In [4]:
class NeuDataset(tf.data.Dataset):
    def read_file_in_batches(num_samples):
        time.sleep(0.03)
        for sample_idx in range(num_samples):
            time.sleep(0.015)
            yield (sample_idx,)
    def __new__(cls, num_samples=3):
        return tf.data.Dataset.from_generator(
            cls.read_file_in_batches,
            output_signature = tf.TensorSpec(shape = (1,), dtype = tf.int64),
            args=(num_samples,))
def benchmark(dataset, num_epochs=2):
    for epoch_num in range(num_epochs):
        for sample in dataset:
            time.sleep(0.01)

In [26]:
%%timeit
benchmark(NeuDataset())

273 ms ± 10.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit
benchmark(NeuDataset().prefetch(1))

293 ms ± 34.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit
benchmark(NeuDataset().prefetch(2))

302 ms ± 34.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%%timeit
benchmark(NeuDataset().prefetch(3))

299 ms ± 79.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit
benchmark(NeuDataset().prefetch(tf.data.AUTOTUNE))

264 ms ± 4.72 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
dataset = tf.data.Dataset.range(5)
dataset = dataset.map(lambda x: x**2)
dataset = dataset.cache("mycache.txt")
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [23]:
def mapped_function(s):
    tf.py_function(lambda: time.sleep(0.3), [], ())
    return s

In [25]:
%%timeit -r1 -n1
benchmark(NeuDataset().map(mapped_function), 5)

5.15 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [24]:
%%timeit -r1 -n1
benchmark(NeuDataset().map(mapped_function).cache(), 5)

1.34 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
